In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import random
from typing import Self
from jaxtyping import Array, Float, Scalar, Key
from tqdm import tqdm
import matplotlib.pyplot as plt
import scipy as sp
import numpy as np
from src import gp, test_functions, acquisition
from einops.array_api import rearrange
from src.gp import *
from src.acquisition import *

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import warnings

jax.config.update("jax_enable_x64", True)

In [ ]:
D = 2
K = 4
N = 2*K
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
NAIVE_PARAMETRIZATION = False

# Target Function

In [ ]:
rkhs_kernel_metric = kernels.Euclidean(log_scale=jnp.log(1 / 100))
rkhs_kernel_profile = kernels.SquaredExponential()


class RKHSFunction(NamedTuple):
    # f = sum ai k(xi, .)
    # y = K(x, x) @ a
    x: Float[Array, "k d"]
    y: Float[Array, "k"]

    @classmethod
    def sample(cls, n: int, k: int, d: int) -> list[Self]:
        lhs_sampler = sp.stats.qmc.LatinHypercube(
            d=k * (d + 1), seed=np.random.mtrand._rand
        )
        x = lhs_sampler.random(n=n).reshape(n, k, d + 1)
        return [cls.from_array(xi) for xi in x]

    @classmethod
    def from_array(cls, x: Float[Array, "k (d+1)"]) -> Self:
        x, y = x[:, :-1], x[:, -1]
        y = 2 * y - 1  # scale to [-1, 1]
        return RKHSFunction(x=x, y=y)

    @eqx.filter_jit
    def evaluate(self, t: Float[Array, "d"]) -> Scalar:
        # compute the kernel vector (x vs t)
        dist_fn = jax.vmap(rkhs_kernel_metric, in_axes=(0, None))
        Ktx = rkhs_kernel_profile(dist_fn(self.x, t))  # (k,)
        if NAIVE_PARAMETRIZATION:
            return Ktx @ self.y
        # compute the kernel matrix (x vs x)
        dist_fn = jax.vmap(dist_fn, in_axes=(None, 0))
        Kxx = rkhs_kernel_profile(dist_fn(self.x, self.x))  # (k, k)
        return Ktx @ jnp.linalg.solve(Kxx, self.y)


@eqx.filter_jit
def target_fn(x: Float[Array, "d"]) -> Scalar:
    d = jnp.linalg.norm(x - 0.5)
    return jnp.sinc(d * 2 * jnp.pi).squeeze()
    return jnp.sin(d * 2 * jnp.pi)


def loss_fn(f: RKHSFunction, lims=(0.0, 1.0)) -> Scalar:
    if D == 1:
        delta = jax.jit(lambda t: jnp.square(f.evaluate(t) - target_fn(t)).squeeze())
        y, error = sp.integrate.quad(delta, *lims)
        return y
    elif D == 2:
        delta = jax.jit(
            lambda t1, t2: jnp.square(
                f.evaluate(jnp.array([t1, t2])) - target_fn(jnp.array([t1, t2]))
            ).squeeze()
        )
        y, error = sp.integrate.dblquad(
            delta, *lims, lambda _: lims[0], lambda _: lims[1]
        )
        return y
    else:
        raise NotImplementedError("Only D=1 and D=2 are supported")

In [ ]:
fs = RKHSFunction.sample(n=N, k=K, d=D)


def plot(fs: list[RKHSFunction]):
    if D == 1:
        x_grid = jnp.linspace(0, 1, 1000).reshape(-1, 1)
        plt.plot(x_grid, jax.vmap(target_fn)(x_grid), color="black", label="target")
        for i, fi in enumerate(fs):
            y = jax.vmap(fi.evaluate)(x_grid)
            plt.plot(x_grid, y, alpha=0.5)
            plt.plot(fi.x, fi.y, "x", color=plt.gca().lines[-1].get_color(), alpha=0.7)
        plt.grid()
        plt.legend()
        plt.xlim(0.0, 1.0)
        plt.show()
    elif D == 2:
        x_grid = jnp.linspace(0, 1, 100)
        y_grid = jnp.linspace(0, 1, 100)
        xx, yy = jnp.meshgrid(x_grid, y_grid)
        zz_f = jax.vmap(lambda t: fs[0].evaluate(t))(
            jnp.stack([xx.ravel(), yy.ravel()], axis=-1)
        ).reshape(xx.shape)
        zz_t = jax.vmap(lambda t: target_fn(t))(
            jnp.stack([xx.ravel(), yy.ravel()], axis=-1)
        ).reshape(xx.shape)

        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.title("Candidate")
        plt.contourf(xx, yy, zz_f, levels=50, cmap="viridis")
        plt.scatter(
            fs[0].x[:, 0], fs[0].x[:, 1], c=fs[0].y, cmap="viridis", edgecolor="black"
        )
        plt.colorbar()

        plt.subplot(1, 2, 2)
        plt.title("Target Function")
        plt.contourf(xx, yy, zz_t, levels=50, cmap="viridis")
        plt.colorbar()

        plt.tight_layout()
        plt.show()
    else:
        raise NotImplementedError("Only D=1 and D=2 are supported")


plot(fs)

In [ ]:
class MaximumMeanDiscrepancy(kernels.Metric[RKHSFunction]):
    rkhs_metric: kernels.Metric[Float[Array, "d"]] = eqx.field(static=True)
    rkhs_profile: kernels.Profile = eqx.field(static=True)
    log_scale: Scalar | None = None

    def initialize(self, xs: list[RKHSFunction]) -> Self:
        return self.replace(log_scale=jnp.zeros(()))

    def __call__(self, x1: RKHSFunction, x2: RKHSFunction) -> Scalar:
        kernel = lambda a, b: self.rkhs_profile(self.rkhs_metric(a, b))
        kernel = jax.vmap(jax.vmap(kernel, in_axes=(None, 0)), in_axes=(0, None))
        K11 = kernel(x1.x, x1.x)
        K12 = kernel(x1.x, x2.x)
        K22 = kernel(x2.x, x2.x)

        if NAIVE_PARAMETRIZATION:
            a1 = x1.y
            a2 = x2.y
        else:
            a1 = jnp.linalg.solve(K11, x1.y)
            a2 = jnp.linalg.solve(K22, x2.y)

        d2 = +a1 @ K11 @ a1 + a2 @ K22 @ a2 - 2 * a1 @ K12 @ a2

        d = jnp.sqrt(d2 + EPS)
        if self.log_scale is not None:
            scale = jnp.exp(self.log_scale)
            d = d / scale
        return d


class BFGS(AcquisitionStrategy[RKHSFunction]):
    multi_starts: int
    raw_samples: int = 32
    max_iterations: int = 100
    ftol: float = EPS
    gtol: float = 0.0

    def __call__(
        self, model: gp.GaussianProcess[RKHSFunction], q: int = 1
    ) -> list[RKHSFunction]:
        k, d = model.observed_xs[0].x.shape

        # define the acquisition function and its gradient as a vect
        @jax.jit
        @jax.value_and_grad
        def loss(x: Float[Array, "k*(d+1)"]):
            f = RKHSFunction.from_array(x.reshape(k, d + 1))
            return -model.log_expected_improvement(f)

        def verbose_loss(x: Float[Array, "k*(d+1)"]):
            val, grad = loss(x)
            if jnp.isnan(val):
                warnings.warn("Warning: NaN encountered in loss!")
            if jnp.isnan(grad).any():
                warnings.warn("Warning: NaN encountered in gradient!")
            return val, grad

        # sample initial guess for candidate points
        lhs_sampler = sp.stats.qmc.LatinHypercube(
            d=k * (d + 1), seed=np.random.mtrand._rand
        )
        x = lhs_sampler.random(n=self.raw_samples)  # (n, k * (d+1))

        # keep the most promising initial guesses
        losses = np.array([loss(xi)[0] for xi in x])
        best_idx = np.argsort(losses)[: self.multi_starts]
        x = x[best_idx]

        # optimize each initial guess with L-BFGS
        results = [
            sp.optimize.minimize(
                fun=verbose_loss,
                x0=xi,
                jac=True,
                method="L-BFGS-B",
                bounds=[(0, 1) for _ in range(k * (d + 1))],
                options=dict(
                    maxiter=self.max_iterations,
                    ftol=self.ftol,
                    gtol=self.gtol,
                ),
            )
            for xi in x
        ]

        # sort results and return best q
        x = np.array([result.x.reshape(k, d + 1) for result in results])
        losses = np.array([result.fun for result in results])
        best_idx = np.argsort(losses)[:q]
        return [RKHSFunction.from_array(xi) for xi in x[best_idx]]

# Fit GP on initial points

In [ ]:
model = gp.GaussianProcess(
    kernel_metric=MaximumMeanDiscrepancy(rkhs_kernel_metric, rkhs_kernel_profile),
    kernel_profile=kernels.SquaredExponential(),
)

x0 = RKHSFunction.sample(n=N, k=K, d=D)
y0 = jnp.array([loss_fn(xi) for xi in x0])
model = model.fit(x0, y0)

In [ ]:
if K == 1 and D == 1:
    # precompute the loss landscape for visualization
    x_grid = jnp.linspace(-5, 5, 30)
    a_grid = jnp.linspace(-5, 5, 30)
    x_meshgrid, a_meshgrid = jnp.meshgrid(x_grid, a_grid)
    fs_meshgrid = jnp.stack([x.meshgrid.flatten(), a_meshgrid.flatten()], axis=-1)
    fs_meshgrid = [RKHSFunction.from_array(fi) for fi in fs_meshgrid]
    losses_meshgrid = jnp.array([loss_fn(fi) for fi in fs_meshgrid])

In [ ]:
if K == 1 and D == 1:
    pred = [model.predict([fi]) for fi in fs_meshgrid]
    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "scene"}, {"type": "xy"}]],
        subplot_titles=("GP Prediction", "GP Uncertainty"),
    )

    # Left subplot: 3D GP mean + true surface + sampled points
    fig.add_trace(
        go.Surface(
            z=jnp.array([p.mean for p in pred]).reshape(x_meshgrid.shape),
            x=x_meshgrid,
            y=a_meshgrid,
            showscale=False,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Surface(
            z=losses_meshgrid.reshape(x_meshgrid.shape),
            x=x_meshgrid,
            y=a_meshgrid,
            opacity=0.3,
            showscale=False,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter3d(
            z=model.observed_ys,
            x=jnp.array([el.x.squeeze() for el in model.observed_xs]),
            y=jnp.array([el.a.squeeze() for el in model.observed_xs]),
            mode="markers",
            marker=dict(color="red", size=5),
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    # Right subplot: uncertainty contour + sampled points
    fig.add_trace(
        go.Contour(
            z=jnp.sqrt(jnp.array([p.cov for p in pred]).reshape(x_meshgrid.shape)),
            x=x_grid,
            y=a_grid,
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=jnp.array([el.x.squeeze() for el in model.observed_xs]),
            y=jnp.array([el.a.squeeze() for el in model.observed_xs]),
            mode="markers",
            marker=dict(color="red", size=10),
            showlegend=False,
        ),
        row=1,
        col=2,
    )
    fig.update_layout(width=800, height=400)
    fig.show()

# Expected Improvement

In [ ]:
acquisition = BFGS(multi_starts=256, raw_samples=1024)
#acquisition = Voronoi(multi_starts=min(1000, K**2))
(best,) = acquisition(model)
print("Best candidate:")
print("x:", best.x.squeeze())
print("y:", best.y.squeeze())

In [ ]:
if K == 1 and D == 1:
    ei = jnp.exp(jnp.array([model.log_expected_improvement(fi) for fi in fs_meshgrid]))
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Expected Improvement", "Surrogate Model"),
    )

    # Left: Expected Improvement
    fig.add_trace(
        go.Contour(
            z=ei.reshape(x_meshgrid.shape),
            x=x_grid,
            y=a_grid,
            colorbar=dict(title="EI"),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[best.x.squeeze()],
            y=[best.a.squeeze()],
            mode="markers",
            marker=dict(color="red", size=10),
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    # Right: Surrogate Model mean
    fig.add_trace(
        go.Contour(
            z=jnp.array([p.mean for p in pred]).reshape(x_meshgrid.shape),
            x=x_grid,
            y=a_grid,
            colorbar=dict(title="Mean"),
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=jnp.array([el.x.squeeze() for el in model.observed_xs]),
            y=jnp.array([el.a.squeeze() for el in model.observed_xs]),
            mode="markers",
            marker=dict(color="red", size=10),
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    fig.update_layout(width=800, height=400, title="Acquisition vs Surrogate")
    fig.show()

# Putting it all together

In [ ]:
models = [model]
candidates = []
for i in (pbar := tqdm(range(500))):
    xs = acquisition(models[-1])
    ys = jnp.array([loss_fn(xi) for xi in xs])

    model = models[-1].fit(
        xs=models[-1].observed_xs + xs,
        ys=jnp.concatenate([models[-1].observed_ys, ys], axis=0),
    )
    models.append(model)
    pbar.set_postfix(loss=model.observed_ys.min())

    best_idx = jnp.argmin(model.observed_ys)
    if i == 0 or (best_idx == len(model.observed_ys) - 1):
        plot([model.observed_xs[best_idx]])